# The 2025 test half, scored once

*Text shortened on 23 September; code and outputs are unchanged. The plan as committed before running is in commit `63e6f25`.*

Every earlier number came from the learning half, where settings were chosen after seeing the
fold scores, so 61.1% is a little optimistic. This scores the frozen model once on the 516 maps
after Masters Toronto, which no model had been scored on. **Committed to git before it ran.**

**Decided in advance:**

| | |
|---|---|
| Model | `make_final_model()`, fitted on all 756 learning maps, unchanged whatever happens |
| Inputs | Player form over the whole season, backwards only (checked below) |
| Measures | Accuracy and log loss on all 516 maps; no breakdowns |
| Compared with | The picker rule (the bar), always team A, and the agent-picks model |
| Margin of error | 5,000 resamples of whole matches, seed 0 |
| Verdict | "Beats the baseline" only if the bottom of the lead's 95% range is above zero; if the range includes zero, "ahead, but not distinguishable" |

Expected: lower than 61.1%, since some of that figure is selection.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)

from dataset import build_map_table, split_season
from features import build_features, feature_columns
from form import player_form
from model import FINAL_COLUMNS, make_final_model, make_model

SEED, RESAMPLES = 0, 5000

## Build the inputs, and check nothing leaks backwards

In [2]:
maps = build_map_table()
learn, test = split_season(maps)
columns = feature_columns(maps)

form_season = player_form(maps, half_life=5)     # whole season, in date order, backwards only
form_learn = player_form(learn, half_life=5)     # learning half alone, as every notebook so far

# The guard: adding the test half must not change a single learning-half value.
pd.testing.assert_frame_equal(form_season.loc[form_learn.index, form_learn.columns], form_learn)
print("learning-half form is identical either way -- extending to the test half leaked nothing backwards")

x_learn, y_learn = build_features(learn, columns)
x_test, y_test = build_features(test, columns)
X_learn = pd.concat([x_learn, form_season.loc[learn.index]], axis=1)
X_test = pd.concat([x_test, form_season.loc[test.index]], axis=1)

print(f"learning maps {len(X_learn)}, test maps {len(X_test)} from {test['match_id'].nunique()} matches, "
      f"{test['played_at'].min():%d %b} to {test['played_at'].max():%d %b %Y}")
print(f"test maps with no rating form on a side: {X_test['rating_diff'].isna().sum()}, "
      f"of which combat score can stand in on {(X_test['rating_diff'].isna() & X_test['acs_diff'].notna()).sum()}")

learning-half form is identical either way -- extending to the test half leaked nothing backwards
learning maps 756, test maps 516 from 202 matches, 03 Jul to 05 Oct 2025
test maps with no rating form on a side: 0, of which combat score can stand in on 0


## Fit on the learning half, predict the test half

In [3]:
final = make_final_model().fit(X_learn[FINAL_COLUMNS].astype(float), y_learn)
agent_model = make_model(LogisticRegression(max_iter=5000)).fit(x_learn.astype(float), y_learn)

chance = {
    "final model": final.predict_proba(X_test[FINAL_COLUMNS].astype(float))[:, 1],
    "agent-picks model": agent_model.predict_proba(x_test.astype(float))[:, 1],
}
guess = {name: (p >= 0.5).astype(int) for name, p in chance.items()}
guess["picker rule"] = test["map_picked_by"].map({"team_a": 1, "team_b": 0}).fillna(1).astype(int).to_numpy()
guess["always team A"] = np.ones(len(test), dtype=int)

truth = y_test.to_numpy()
right = {name: (g == truth) for name, g in guess.items()}

## The result

In [4]:
results = pd.DataFrame({
    name: {"accuracy": right[name].mean(),
           "log loss": log_loss(truth, chance[name], labels=[0, 1]) if name in chance else np.nan}
    for name in right
}).T
print(f"{len(truth)} test maps.  Log loss for saying 50/50 to every map: 0.693\n")
print(results.to_string(formatters={"accuracy": "{:.1%}".format, "log loss": "{:.3f}".format}))

516 test maps.  Log loss for saying 50/50 to every map: 0.693

                  accuracy log loss
final model          54.5%    0.689
agent-picks model    50.8%    0.704
picker rule          52.9%      NaN
always team A        52.3%      NaN


## The margin of error

In [5]:
rng = np.random.default_rng(SEED)
match_of_map = test["match_id"].to_numpy()
matches = np.unique(match_of_map)
maps_in = {m: np.flatnonzero(match_of_map == m) for m in matches}

accuracy, lead = [], []
for _ in range(RESAMPLES):
    rows = np.concatenate([maps_in[m] for m in rng.choice(matches, size=len(matches), replace=True)])
    accuracy.append(right["final model"][rows].mean())
    lead.append(right["final model"][rows].mean() - right["picker rule"][rows].mean())

acc_lo, acc_hi = np.percentile(accuracy, [2.5, 97.5])
lead_point = right["final model"].mean() - right["picker rule"].mean()
lead_lo, lead_hi = np.percentile(lead, [2.5, 97.5])

print(f"final model accuracy        {right['final model'].mean():.1%}   95% range {acc_lo:.1%} to {acc_hi:.1%}")
print(f"lead over the picker rule   {lead_point * 100:+.1f} points   95% range {lead_lo * 100:+.1f} to {lead_hi * 100:+.1f}")

if lead_point > 0 and lead_lo > 0:
    verdict = "BEATS THE BASELINE"
elif lead_point > 0:
    verdict = "AHEAD, BUT NOT DISTINGUISHABLE FROM THE BASELINE ON THIS MANY MAPS"
else:
    verdict = "DOES NOT BEAT THE BASELINE"
print(f"\nby the rule decided in advance: {verdict}")

final model accuracy        54.5%   95% range 50.3% to 58.7%
lead over the picker rule   +1.6 points   95% range -3.2 to +6.4

by the rule decided in advance: AHEAD, BUT NOT DISTINGUISHABLE FROM THE BASELINE ON THIS MANY MAPS


## What this means

*Written after the cells above ran, in a separate commit.*

**54.5% on 516 unseen maps, against the picker rule's 52.9%:** a lead of 1.6 points (range −3.2
to +6.4). **By the rule set in advance: ahead, but not distinguishable from the baseline.**

- **Agent picks came last again** (50.8%), with a log loss of 0.704, no better than 50/50 (0.693).
- **Everything fell:** the model 61.1% → 54.5%, the picker rule 55.5% → 52.9%. The lead fell from
  5.6 points to 1.6, but its range still includes 5.6, so the test can't tell a real shrink from
  luck.
- **Not a bug.** The test inputs look like the learning inputs (checked without reading any
  result; `check_test_inputs.py` reproduces this):

| | Learning half | Test half |
|---|---|---|
| Rating gap, average (should be about 0) | +0.009 | −0.008 |
| Rating gap, spread | 0.101 | 0.092 |
| Rating gap, median size | 0.064 | 0.056 |
| Maps with no rating form on a side | 10.7% | 0% |
| Deciders | 17.1% | 18.6% |
| Model's typical distance from 50% | 5.8 points | 6.6 points |
| Share of maps called for team A | 49.2% | 46.7% |

Why the edge looks smaller isn't investigated: that would mean slicing the test half after
seeing it. 2026's sealed results are the place to find out.